# Full Pipeline on Real OmniDocBench Data

End-to-end Colab notebook for the adaptive-inference research MVP. Goes from a fresh
Colab VM to:

1. Repo + deps installed.
2. Real OmniDocBench English-table subset downloaded and filtered.
3. Phase 1 manifests built (20-page calibration split).
4. Phase 2 single-pass baseline run + scored (real InternVL2-2B).
5. Phase 4 adaptive run + scored.
6. **Phase 5 calibration sweep + frozen budgets** (the step we are currently on).
7. Important artifacts copied to Google Drive so they survive a tab close.

Total wall time on Colab Pro: roughly 25–40 minutes the first time (model download +
calibration sweep dominate).

## ⚠️ Important: how Colab persistence works (read this once)

Colab runs on a remote VM that gets wiped when:
- you close the browser tab,
- the runtime is idle for ~90 minutes,
- the runtime is reset, or
- 12 hours pass (free) / 24 hours (Pro).

Wiped means: every file under `/content/`. Your code, downloaded data, run outputs,
model weights cache — gone. Colab **does not auto-push to GitHub**. The recovery rules:

| What | Where it lives | Recovery |
|---|---|---|
| **Source code** | GitHub | `git clone` re-pulls. Always commit + push from your Mac before closing. |
| **Real data** (`data/omnidocbench/`) | gitignored, ephemeral | Re-run the downloader (~1 min). |
| **Run artifacts** (`outputs/runs/...`) | gitignored, ephemeral | Re-run the model (slow). Save to Drive instead. |
| **Frozen budgets** (`configs/calibration/frozen_budgets.json`) | checked into git | Save to Drive **and** download → push to GitHub from Mac. |
| **Model weights cache** (HF cache) | ephemeral | Re-download (~1 min). |

**Rules of thumb:**
1. Edit code on your Mac, commit/push to GitHub, then `git pull` in Colab. Never edit
   code directly in Colab files — that work is unbacked.
2. Run **Section 7 (Save to Drive)** before closing the tab. Always.
3. For artifacts that belong in the repo (like `frozen_budgets.json`), download them
   from Drive to your Mac and commit from there.
4. Click **Runtime → Disconnect and delete runtime** when truly done; the VM goes away
   and you free GPU quota.

## Loading this notebook into Colab from GitHub

Two equivalent ways:

**Option A — direct URL** (fastest):
```
https://colab.research.google.com/github/Michaelhamaty/Resarch_dev/blob/main/notebooks/colab_full_pipeline_real.ipynb
```
Bookmark this. It always opens the latest committed version of the notebook.

**Option B — Colab UI:** File → Open notebook → **GitHub** tab → paste
`Michaelhamaty/Resarch_dev` → pick the notebook from the list.

Either way, the first thing the notebook does is `git clone` (or `git pull`) the repo,
so you always have the latest code regardless of how stale the notebook view is.

**Make sure** before you open this in Colab: from your Mac, `git status` is clean
and `git log origin/main..HEAD` is empty — i.e. all your local commits are pushed.

## Section 0 — Setup (clone repo, install deps)

In [ ]:
# Clone (or pull) the repo. Idempotent: re-running is safe.
%cd /content
import os
if not os.path.isdir("Research_claude/.git"):
    !git clone https://github.com/Michaelhamaty/Resarch_dev.git Research_claude
else:
    !cd Research_claude && git pull origin main
%cd /content/Research_claude
!git log --oneline -5

In [ ]:
# Install project (editable) + missing deps. Colab preinstalls torch + transformers.
!pip install -q huggingface_hub einops timm
!pip install -q -e .

In [ ]:
# Verify GPU and key deps. If CUDA is False, switch runtime to GPU
# (Runtime → Change runtime type → GPU) before continuing.
import torch, sys
print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU:     {torch.cuda.get_device_name(0)}")
    print(f"  Compute: sm_{''.join(str(x) for x in torch.cuda.get_device_capability(0))}")

## Section 1 — Build the OmniDocBench data fixture

Downloads the OmniDocBench annotations + page images for up to 50 English-table pages
with at least 8 non-empty gold cells (the filter we added to drop slide-layout strips
and half-empty survey templates). Produces:

- `data/omnidocbench/images/<page_id>.<ext>` — raw page images.
- `data/omnidocbench/records.json` — page records.
- `data/omnidocbench/ground_truth.json` — `{page_id: gold_html}` mapping.

Then runs the Phase 1 manifest builder, which writes the fixed calibration split
(20 pages, per `configs/dataset/phase1_omnidocbench.yaml`).

In [ ]:
%cd /content/Research_claude
!python scripts/data/build_omnidocbench_fixture.py --limit 50 --min-non-empty-cells 8

In [ ]:
!python scripts/subset_extraction/build_phase1_manifests.py \
    --config configs/dataset/phase1_omnidocbench.yaml

In [ ]:
# Sanity check: print the calibration split with table dimensions.
import json
from pathlib import Path
RECS = {r["page_id"]: r for r in json.loads(Path("data/omnidocbench/records.json").read_text())}
SPLIT = json.loads(Path("data/splits/omnidocbench/calibration_split.json").read_text())
print(f"records.json total pages: {len(RECS)}")
print(f"calibration_split pages : {len(SPLIT['page_ids'])}")
print()
for pid in SPLIT["page_ids"]:
    r = RECS[pid]
    print(f"  rows={r['row_count']:3d} cols={r['col_count']:2d}  {pid}")

## Section 2 — Phase 2 baseline (single-pass low budget)

Real InternVL2-2B at `max_tiles=4`, prompt `table_parse_v2`. This is the
fixed-low baseline. First run will download the 4.4 GB model checkpoint —
subsequent runs reuse the HF cache.

Expect macro_cell_f1 around 0.027 and ~10/20 pages with `no_tables` parse errors
on this calibration split (those are the pages the verifier will flag for reparse
in the adaptive run).

In [ ]:
# Wipe any stale outputs so we don't confuse old vs new.
!rm -rf outputs/runs/colab_real_2b_omnidocbench_low_v1 outputs/analysis/score_real_v1

!python scripts/main_runs/run_single_pass.py \
    --config configs/runs/colab_real_2b_omnidocbench_low.yaml

In [ ]:
!python scripts/analysis/score_run.py \
    --run-dir outputs/runs/colab_real_2b_omnidocbench_low_v1 \
    --ground-truth data/omnidocbench/ground_truth.json \
    --output-dir outputs/analysis/score_real_v1

print("=" * 70); print("BASELINE SUMMARY (single-pass low budget)"); print("=" * 70)
!cat outputs/analysis/score_real_v1/summary.md

## Section 3 — Phase 4 adaptive (verifier-gated reparse to high budget)

Same 20 pages, low budget `max_tiles=4`, escalation budget `max_tiles=12`. Verifier
structurally inspects each first-pass output; pages that fail trigger a one-shot
reparse at the high budget.

Expect macro_cell_f1 around 0.064 and ~4/20 remaining `no_tables` after reparse.
Compute is roughly 1.5× the baseline because of the reparses.

In [ ]:
!rm -rf outputs/runs/colab_real_2b_omnidocbench_adaptive_v1 outputs/analysis/score_real_adaptive_v1

!python scripts/main_runs/run_adaptive.py \
    --config configs/runs/colab_real_2b_omnidocbench_adaptive.yaml

In [ ]:
!python scripts/analysis/score_run.py \
    --run-dir outputs/runs/colab_real_2b_omnidocbench_adaptive_v1 \
    --ground-truth data/omnidocbench/ground_truth.json \
    --output-dir outputs/analysis/score_real_adaptive_v1

print("=" * 70); print("ADAPTIVE SUMMARY"); print("=" * 70)
!cat outputs/analysis/score_real_adaptive_v1/summary.md

## Section 4 — Phase 5 calibration (the step we are currently on)

Sweeps candidate budgets on the 20-page calibration split, picks `(B_low, B_high,
B_fix_2B, B_fix_8B)` matched at average compute, and writes the result to
`configs/calibration/frozen_budgets.json`. This artifact is the input to Phase 6.

**Compute:** ~205 real InternVL2-2B inferences (~10–15 minutes on Colab Pro). Sweep
config: 4 adaptive `(low, high)` pairs + 4 fixed-2B candidates + 3 fixed-8B candidates.
8B is stub-only — its frozen budget will carry `adapter_kind: stub` and Phase 6 will
skip it via `--allow-stubbed-8b`.

**Prerequisite:** `configs/calibration/phase5_omnidocbench.yaml` must be on the
default branch. If `git log --oneline -5` from Section 0 doesn't show the
`Add real-data Phase 5 calibration config` commit, push it from your Mac before
running this section.

In [ ]:
# Wipe stale sweep state so an earlier partial run doesn't taint the artifact.
!rm -rf outputs/calibration/sweep_omnidocbench outputs/calibration/sweep_omnidocbench_summaries.jsonl

!python scripts/calibration/run_calibration.py \
    --config configs/calibration/phase5_omnidocbench.yaml

In [ ]:
# Inspect the frozen budgets that just got picked.
import json
from pathlib import Path

frozen = json.loads(Path("configs/calibration/frozen_budgets.json").read_text())

print(f"run_id           : {frozen['run_id']}")
print(f"generated_at     : {frozen['generated_at']}")
print()
print("FROZEN BUDGETS")
for name, b in frozen["budgets"].items():
    print(f"  {name:10s}  max_tiles={b['max_tiles']:3d}  "
          f"model={b['model_name']:14s}  adapter={b['adapter_kind']}")
print()
print("SELECTION")
sel = frozen["selection"]
print(f"  adaptive   : low={sel['adaptive']['low_max_tiles']} "
      f"high={sel['adaptive']['high_max_tiles']}  "
      f"target={sel['adaptive']['target_cost_tiles']:.2f}  "
      f"measured={sel['adaptive']['measured_cost_tiles']:.2f}")
print(f"  fixed_2b   : max_tiles={sel['fixed_2b']['max_tiles']}  "
      f"target={sel['fixed_2b']['target_cost_tiles']:.2f}  "
      f"measured={sel['fixed_2b']['measured_cost_tiles']:.2f}  "
      f"within_tolerance={sel['fixed_2b']['within_tolerance']}")
print(f"  fixed_8b   : max_tiles={sel['fixed_8b']['max_tiles']}  "
      f"target={sel['fixed_8b']['target_cost_tiles']:.2f}  "
      f"measured={sel['fixed_8b']['measured_cost_tiles']:.2f}  "
      f"within_tolerance={sel['fixed_8b']['within_tolerance']}")

In [ ]:
# Full sweep table — every (budget, cost, reparse_rate) point we measured.
import json
print(f"{'sweep':10s}  {'tiles':14s}  {'cost':>6s}  {'reparse_rate':>12s}")
print("=" * 60)
with open("outputs/calibration/sweep_omnidocbench_summaries.jsonl") as f:
    for line in f:
        d = json.loads(line)
        kind = d.get("sweep_kind", "?")
        if kind == "adaptive":
            tiles = f"low={d['low_max_tiles']} high={d['high_max_tiles']}"
            rep = f"{d.get('reparse_rate', 0.0):.2f}"
        else:
            tiles = f"max={d['max_tiles']}"
            rep = "\u2014"
        print(f"{kind:10s}  {tiles:14s}  {d['cost_tiles']:6.2f}  {rep:>12s}")

## Section 5 — Save artifacts to Google Drive

**Run this before closing the tab.** Mounts your Drive and copies the most
important artifacts to `MyDrive/research_claude_session/`. Drive persists across
Colab sessions; `/content/` does not.

On a fresh notebook session you can pull the saved artifacts back if you want
to skip re-running the model — see Section 6.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/research_claude_session")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

TO_SAVE = [
    "outputs/analysis",
    "outputs/calibration",
    "outputs/runs",
    "configs/calibration/frozen_budgets.json",
    "data/omnidocbench/records.json",
    "data/omnidocbench/ground_truth.json",
    "data/splits/omnidocbench",
]
for src in TO_SAVE:
    src_path = Path(src)
    if not src_path.exists():
        print(f"  SKIP {src} (not found on disk)")
        continue
    dst = DRIVE_DIR / src
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src_path.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src_path, dst)
    else:
        shutil.copy2(src_path, dst)
    print(f"  saved {src}")
print()
print(f"Drive contents now at {DRIVE_DIR}:")
!ls -la /content/drive/MyDrive/research_claude_session/

## Section 6 — Recovering from Drive on a fresh Colab session

If you reopen this notebook in a new Colab session and don't want to re-run the model
(but the cloned repo on `/content/` is fresh and empty), you can pull saved artifacts
from Drive into the freshly cloned repo. Run after Section 0:

In [ ]:
# Optional — only run this if you want to skip re-running the pipeline.
RESTORE_FROM_DRIVE = False

if RESTORE_FROM_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    from pathlib import Path
    SRC = Path("/content/drive/MyDrive/research_claude_session")
    DST = Path("/content/Research_claude")
    for sub in ["outputs", "data", "configs/calibration/frozen_budgets.json"]:
        s = SRC / sub
        d = DST / sub
        if not s.exists():
            print(f"  SKIP {sub} (not in Drive)")
            continue
        d.parent.mkdir(parents=True, exist_ok=True)
        if s.is_dir():
            if d.exists():
                shutil.rmtree(d)
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
        print(f"  restored {sub}")

## Section 7 — Commit `frozen_budgets.json` back to GitHub

`frozen_budgets.json` is **checked into the repo** — it's the contract Phase 6 reads.
After Phase 5 regenerates it, you need to land it on `main`. Colab can't push to
GitHub for you (no credentials), so the workflow is:

1. Make sure the most recent Section 5 (Save to Drive) ran successfully.
2. On your Mac, sync from Drive (use the Google Drive desktop client, or download
   the file directly from `https://drive.google.com/`).
3. Copy the file into your local repo, replacing the existing one:
   ```bash
   cp ~/Google\ Drive/MyDrive/research_claude_session/configs/calibration/frozen_budgets.json \
      /Users/michaelhamaty/Developer/Research_claude/configs/calibration/frozen_budgets.json
   ```
4. Commit and push:
   ```bash
   cd /Users/michaelhamaty/Developer/Research_claude
   git status                           # confirm only frozen_budgets.json changed
   git diff configs/calibration/frozen_budgets.json   # eyeball the diff
   git add configs/calibration/frozen_budgets.json
   git commit -m "Refreeze budgets from real Phase 5 calibration on OmniDocBench"
   git push origin main
   ```

After that push, Phase 6 (the next step in the project) will read these real budgets
instead of the stub placeholders.

## What comes next (after this notebook completes)

Once Section 4 has produced sane frozen budgets and Section 7 has landed them on
GitHub, the next step in the research is **Phase 6** — run the five systems
(adaptive, fixed_2b_low, fixed_2b_matched, random_2b × 2 seeds, fixed_8b_matched)
against the same calibration split using the frozen budgets, then **Phase 7** to
produce the comparative analysis tables.

We'll plan and add those sections in their own commit. Don't try to run them in this
notebook yet — the configs and CLI flags need a small audit against the real-data
path first.